# Notebook 2: Data Cleaning & Mention Share Computation

Combines GDELT and Ngrams raw data, normalizes across sources, and computes `mention_share`.

**Key output:** `data/processed/mention_share.csv` — one row per candidate-election,
with columns: `year`, `candidate`, `party`, `mention_share`, `source`.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW_DIR = Path('../data/raw')
PROCESSED_DIR = Path('../data/processed')
PROCESSED_DIR.mkdir(exist_ok=True)

In [ ]:
# Load raw data from both sources
gdelt = pd.read_csv(RAW_DIR / 'gdelt_mentions.csv')
ngrams = pd.read_csv(RAW_DIR / 'ngrams_mentions.csv')

df_raw = pd.concat([gdelt, ngrams], ignore_index=True)
print(f'Total rows: {len(df_raw)}')
print(df_raw.groupby('year')[['candidate', 'raw_mentions']].apply(lambda x: x.set_index('candidate')['raw_mentions'].to_dict()))

In [ ]:
# Compute mention_share per election year
# Denominator = sum of ALL candidates (including third-party) per year
# Per pre-registration: third parties included in denominator but excluded from H1

df_raw['mention_share'] = df_raw.groupby('year')['raw_mentions'].transform(
    lambda x: x / x.sum()
)

print(df_raw.groupby('year')[['candidate', 'mention_share']].apply(
    lambda x: x.set_index('candidate')['mention_share'].round(3).to_dict()
))

In [ ]:
# Load official vote share data
vote = pd.read_csv(PROCESSED_DIR / 'election_results.csv')

# Merge
df = df_raw.merge(
    vote[['year', 'candidate', 'popular_vote_share', 'popular_vote_winner', 'incumbent_running']],
    on=['year', 'candidate'],
    how='inner'
)

# Flag: is this candidate the attention leader in this election?
df['attention_leader'] = df.groupby('year')['mention_share'].transform(
    lambda x: (x == x.max()).astype(int)
)

# Major party only (exclude third-party rows from analysis df)
df_major = df[df['party'].isin(['D', 'R'])].copy()

df_major.to_csv(PROCESSED_DIR / 'mention_share.csv', index=False)
print('Saved to data/processed/mention_share.csv')
df_major[['year', 'candidate', 'party', 'mention_share', 'popular_vote_share', 
          'popular_vote_winner', 'attention_leader', 'source']].sort_values('year')

In [ ]:
# Validation checks
# 1. mention_share within each year should sum to 1 (across all candidates incl. third party)
share_sums = df.groupby('year')['mention_share'].sum()
assert (share_sums.round(6) == 1.0).all(), f'mention_share does not sum to 1: {share_sums}'

# 2. Exactly one attention_leader per year among major-party candidates
leaders_per_year = df_major.groupby('year')['attention_leader'].sum()
assert (leaders_per_year == 1).all(), f'Multiple attention leaders in some years: {leaders_per_year}'

print('All validation checks passed.')